In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Authentication is done!")

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project="physionet-492612") #Billing/project context

#Reference the dataset in the public project
dataset_ref = client.dataset('mimiciii_clinical', project='physionet-data')

print("Tables of mimiciii_clinical dataset:")
tables = list(client.list_tables(dataset_ref))
for table in tables:
    print(table.table_id)

print("Number of tables of mimiciii_clinical dataset:", len(tables))

In [ ]:
!pip install pyhealth torch scikit-learn pandas tqdm

In [ ]:
!pip list | grep pyhealth

In [ ]:
diagnoses_query = "SELECT * FROM `physionet-data.mimiciii_clinical.diagnoses_icd`"
diagnoses_df = client.query(diagnoses_query).to_dataframe()
#diagnoses_df.head(10)

In [ ]:
procedures_query = "SELECT * FROM `physionet-data.mimiciii_clinical.procedures_icd`"
procedures_df = client.query(procedures_query).to_dataframe()
#procedures_df.head(10)

In [ ]:
prescriptions_query = "SELECT * FROM `physionet-data.mimiciii_clinical.prescriptions`"
prescriptions_df = client.query(prescriptions_query).to_dataframe()
#prescriptions_df.head(10)

In [ ]:
from pyhealth.utils import set_seed

set_seed(42)

In [ ]:
import tempfile
from pyhealth.datasets import MIMIC3Dataset

In [ ]:
from google.cloud import bigquery
import os

client = bigquery.Client(project="physionet-492612")

root_dir = "/content/mimic3_official"
os.makedirs(root_dir, exist_ok=True)

print("Saving dataset to:", root_dir)

# =========================
# CORE TABLES (EVENT DATA)
# =========================

print("Downloading DIAGNOSES_ICD...")
diagnoses_df = client.query("""
SELECT *
FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
""").to_dataframe()

print("Downloading PROCEDURES_ICD...")
procedures_df = client.query("""
SELECT *
FROM `physionet-data.mimiciii_clinical.procedures_icd`
""").to_dataframe()

print("Downloading PRESCRIPTIONS...")
prescriptions_df = client.query("""
SELECT *
FROM `physionet-data.mimiciii_clinical.prescriptions`
""").to_dataframe()

# =========================
# STRUCTURE TABLES (IMPORTANT)
# =========================

print("Downloading PATIENTS...")
patients_df = client.query("""
SELECT *
FROM `physionet-data.mimiciii_clinical.patients`
""").to_dataframe()

print("Downloading ADMISSIONS...")
admissions_df = client.query("""
SELECT *
FROM `physionet-data.mimiciii_clinical.admissions`
""").to_dataframe()

print("Downloading ICUSTAYS...")
icustays_df = client.query("""
SELECT *
FROM `physionet-data.mimiciii_clinical.icustays`
""").to_dataframe()

# =========================
# SAVE EVERYTHING
# =========================

diagnoses_df.to_csv(f"{root_dir}/DIAGNOSES_ICD.csv", index=False)
procedures_df.to_csv(f"{root_dir}/PROCEDURES_ICD.csv", index=False)
prescriptions_df.to_csv(f"{root_dir}/PRESCRIPTIONS.csv", index=False)

patients_df.to_csv(f"{root_dir}/PATIENTS.csv", index=False)
admissions_df.to_csv(f"{root_dir}/ADMISSIONS.csv", index=False)
icustays_df.to_csv(f"{root_dir}/ICUSTAYS.csv", index=False)

print("\nDONE: Full PyHealth-compatible MIMIC-III dataset created.")

In [ ]:
import os

print(os.listdir("/content/mimic3_official"))

In [ ]:
from pyhealth.datasets import MIMIC3Dataset

base_dataset = MIMIC3Dataset(
    root="/content/mimic3_official",
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
    dev=False,
)

base_dataset.stats()

In [ ]:
"""
base_dataset = MIMIC3Dataset(
    root="https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III",
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
    cache_dir=tempfile.mkdtemp(),
    dev=True,
)

base_dataset.stats()
"""

In [ ]:
from pyhealth.tasks import ReadmissionPredictionMIMIC3

task = ReadmissionPredictionMIMIC3(exclude_minors=False)
sample_dataset = base_dataset.set_task(task)

In [ ]:
from pyhealth.datasets import split_by_patient, get_dataloader

train_dataset, val_dataset, test_dataset = split_by_patient(
    sample_dataset, [0.8, 0.1, 0.1]
)

train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
from pyhealth.models import RNN

model = RNN(dataset=sample_dataset)

In [ ]:
from pyhealth.trainer import Trainer

trainer = Trainer(model=model)
trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=1,
    monitor="roc_auc",
)

In [ ]:
trainer.evaluate(test_dataloader)

In [ ]:
results = trainer.evaluate(test_dataloader)

print("\nFinal Test Results:")
print(results)

In [ ]:
import tempfile

from pyhealth.datasets import MIMIC3Dataset, split_by_patient, get_dataloader
from pyhealth.tasks import ReadmissionPredictionMIMIC3
from pyhealth.models import RNN
from pyhealth.trainer import Trainer


if __name__ == "__main__":

    # =========================
    # STEP 1: Load dataset
    # =========================
    cache_dir = tempfile.mkdtemp()

    base_dataset = MIMIC3Dataset(
        root="/content/mimic3_official",
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=cache_dir,
        dev=False,   # FIX: use full dataset for stable class distribution
    )

    base_dataset.stats()


    # =========================
    # STEP 2: Define task
    # =========================
    task = ReadmissionPredictionMIMIC3(exclude_minors=False)
    sample_dataset = base_dataset.set_task(task)


    # =========================
    # STEP 3: Train / Val / Test split
    # =========================
    train_dataset, val_dataset, test_dataset = split_by_patient(
        sample_dataset,
        [0.8, 0.1, 0.1],
        seed=42   # FIX: reproducible and more stable splits
    )

    train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
    val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
    test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)


    # =========================
    # STEP 4: Define model
    # =========================
    model = RNN(
        dataset=sample_dataset
    )


    # =========================
    # STEP 5: Train model
    # =========================
    trainer = Trainer(model=model)

    trainer.train(
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        epochs=5,
        monitor="roc_auc",   # required metric
    )


    # =========================
    # STEP 6: Evaluate model
    # =========================
    results = trainer.evaluate(test_dataloader)

    print("\nFinal Test Results:")
    print(results)